# pyRiverBed 2.0 — notebook frontend

This notebook is the third way to drive pyRiverBed, alongside the command line
(`pyriverbed run`) and the graphical interface (`pyriverbed gui`). All three
share the same model; the notebook just gives you the model's own objects to
poke at.

**Contents**

1. [Setup](#1)
2. [The simplest possible run](#2)
3. [Looking at the results](#3)
4. [A single cross section — the Beck profile](#4)
5. [What the Kinoshita parameters do](#5)
6. [What the curvature phase lag does](#6)
7. [Mode 2: your own river](#7)
8. [Meander migration and cutoffs](#8)
9. [Chute cutoffs](#9)
10. [Working with the raw arrays](#10)
11. [Saving a configuration for the CLI or GUI](#11)
12. [Optional: an interactive form](#12)

The physics behind every step is documented in
[`THEORY_GUIDE.md`](../THEORY_GUIDE.md).

<a id="1"></a>
## 1. Setup

pyRiverBed needs only NumPy, SciPy and Matplotlib. If you installed it with
`pip install -e .` from the repository root, the import below just works;
otherwise the second line puts the repository on the path.

In [ ]:
# import sys; sys.path.insert(0, '..')     # uncomment if pyRiverBed is not installed

import matplotlib.pyplot as plt
import numpy as np

import pyriverbed as prb
from pyriverbed.notebook import (configure, cross_section, quick_run, show,
                                show_bed, show_curvature, show_diagnostics,
                                show_planform, print_config)

%matplotlib inline
print('pyRiverBed', prb.__version__)

<a id="2"></a>
## 2. The simplest possible run

`quick_run` takes flat keyword arguments, builds a validated configuration and
runs the model. The defaults reproduce the laboratory flume of
[Abad & Garcia (2009)](https://doi.org/10.1029/2008WR007016): a 0.6 m wide,
0.15 m deep Kinoshita channel.

Here we switch the file output off so the notebook directory stays clean — the
results live in the returned object.

In [ ]:
result = quick_run(
    n_bends=3,
    width=0.6,
    depth=0.15,
    save_xyz=False, save_mesh=False, save_bankline=False, save_figures=False,
    log_file='',
)
result.summary()

<a id="3"></a>
## 3. Looking at the results

`show` draws the same three-panel figure that the CLI and GUI write to disk:
the plan view rendered as an exposed-bar visualisation, the curvature signal at
its three stages of processing, and the bed on the curvilinear grid.

In [ ]:
show(result)

The individual panels are available separately, which is usually what you want
in a notebook.

In [ ]:
show_planform(result)
show_curvature(result)
show_bed(result);

<a id="4"></a>
## 4. A single cross section — the Beck profile

This is the heart of the bed model, and it is worth looking at directly. The
flow depth grows **linearly** towards the outer bank, where the bed is a scour
surface cut by a strong secondary current, and decays **exponentially** towards
the inner bank, where it is the depositional face of a point bar. The depth on
the centerline is not free: it is fixed by requiring the cross-sectional area to
stay equal to `width * depth`, so deepening a pool automatically raises the
opposite bar.

In [ ]:
for fraction in (0.25, 0.4, 0.5):
    cross_section(result, fraction)

The bed is deepest against the outer bank of each bend and shallowest against
the inner one, and the two swap over as the channel changes its direction of
turning.

In [ ]:
# Check the area conservation the model relies on, section by section.
channel = result.config.channel
n = np.linspace(-channel.half_width, channel.half_width,
                2 * channel.n_offsets + 1)
depth_profile = channel.depth - result.bed.z
area = np.trapezoid(depth_profile, n, axis=1)

print(f'target area      : {channel.width * channel.depth:.6f} m^2')
print(f'computed area    : {area.mean():.6f} +/- {area.std():.2e} m^2')
print(f'max relative error: {np.max(np.abs(area / (channel.width * channel.depth) - 1)):.2e}')

<a id="5"></a>
## 5. What the Kinoshita parameters do

The Kinoshita curve prescribes the channel's **direction angle** rather than its
coordinates:

$$\theta(s) = \theta_0 \sin\!\left(\frac{2\pi s}{\lambda}\right)
+ \theta_0^3\left[J_s \cos\!\left(\frac{6\pi s}{\lambda}\right)
- J_f \sin\!\left(\frac{6\pi s}{\lambda}\right)\right]$$

With $J_s = J_f = 0$ it is the classical sine-generated curve. The
third-harmonic terms are what make it look like a real river: $J_s$ skews the
bends in the streamwise direction and $J_f$ flattens their apexes.

In [ ]:
from pyriverbed.planform import build_kinoshita
from pyriverbed.config import ChannelConfig, KinoshitaConfig

cases = [
    ('sine-generated ($J_s=J_f=0$)', dict(skewness=0.0, flatness=0.0)),
    ('skewed ($J_s=0.03$)',          dict(skewness=0.03, flatness=0.0)),
    ('flattened ($J_f=0.03$)',       dict(skewness=0.0, flatness=0.03)),
    ('both (pyRiverBed default)',    dict(skewness=0.03125, flatness=0.00520833)),
]

fig, axes = plt.subplots(len(cases), 1, figsize=(11, 9), layout='constrained')
for ax, (title, kwargs) in zip(axes, cases):
    line = build_kinoshita(
        KinoshitaConfig(n_bends=3, arc_wavelength=10.0,
                        max_angular_amplitude=110.0, **kwargs),
        ChannelConfig(ds=0.01))
    ax.plot(line.x, line.y, color='#c0392b', linewidth=1.6)
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_title(f'{title}   —   sinuosity {line.sinuosity:.3f}', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[-1].set_xlabel('x (m)');

Raising the maximum angular amplitude $\theta_0$ is what drives the sinuosity
up, and eventually produces the tight, high-amplitude bends where cutoffs start
to matter.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
for theta0 in (30, 60, 90, 110, 125):
    line = build_kinoshita(
        KinoshitaConfig(n_bends=2, arc_wavelength=10.0,
                        max_angular_amplitude=theta0),
        ChannelConfig(ds=0.01))
    ax.plot(line.x, line.y, linewidth=1.6,
            label=fr'$\theta_0={theta0}^\circ$, $\Omega$={line.sinuosity:.2f}')
ax.set_aspect('equal', adjustable='datalim')
ax.legend(frameon=False)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Effect of the maximum angular amplitude')
ax.spines[['top', 'right']].set_visible(False);

<a id="6"></a>
## 6. What the curvature phase lag does

A real river's deepest point sits **downstream** of the bend apex, because the
secondary flow cell needs a finite distance to spin up and then persists past
the bend exit. pyRiverBed reproduces this by replacing the local curvature with
a linearly decaying, upstream-only weighted average of it, over a window set in
channel widths by `lag_strength`.

Turn the lag off and the pools sit exactly at the apexes — which no river does.

In [ ]:
from pyriverbed.planform import phase_lag

s = np.linspace(0, 60, 3000)
curvature = np.sin(2 * np.pi * s / 15)

fig, ax = plt.subplots(figsize=(11, 4), layout='constrained')
ax.axhline(0, color='0.85', linewidth=0.8)
ax.plot(s, curvature, color='0.4', linewidth=1.2, label='no lag')
for strength, colour in ((2, '#f39c12'), (4, '#e74c3c'), (8, '#8e44ad')):
    window = int(strength * len(s) / s[-1])          # 1 m == 1 channel width here
    ax.plot(s, phase_lag(curvature, window), color=colour, linewidth=1.4,
            label=f'lag_strength = {strength} widths')
ax.set_xlim(10, 60)
ax.set_xlabel('streamwise distance (channel widths)')
ax.set_ylabel('dimensionless curvature')
ax.set_title('The phase lag shifts the signal downstream and damps it')
ax.legend(frameon=False, ncol=2)
ax.spines[['top', 'right']].set_visible(False);

In [ ]:
# The same thing on a real bed: where does the pool sit?
lagged = quick_run(n_bends=3, lag=True, lag_strength=6.0,
                   save_xyz=False, save_mesh=False, save_bankline=False,
                   save_figures=False, log_file='', verbose=False)
unlagged = quick_run(n_bends=3, lag=False,
                     save_xyz=False, save_mesh=False, save_bankline=False,
                     save_figures=False, log_file='', verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 8), layout='constrained')
for ax, run, title in ((axes[0], unlagged, 'no phase lag'),
                       (axes[1], lagged, 'phase lag = 6 widths')):
    ax.imshow(run.bed.z / run.config.channel.depth, cmap='gist_earth',
              aspect=run.config.channel.n_offsets * 16 / run.bed.n_streamwise,
              interpolation='bilinear', vmin=-1, vmax=1)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([0, run.config.channel.n_offsets,
                   2 * run.config.channel.n_offsets], ['-1', '0', '1'])
    ax.set_xlabel('n / half-width')
axes[0].set_ylabel('streamwise node');

<a id="7"></a>
## 7. Mode 2: your own river

Point `centerline_file` at a two-column `x y` text file in any projected
(metric) coordinate system. The repository ships several examples in
`example_centerlines/`.

`smoothing` is the parameter that matters most for a real centerline: curvature
is a second derivative, so digitising noise turns into spurious pools. Raise it
until the curvature signal looks like a sequence of bends rather than noise —
and no further, because over-smoothing flattens the apexes and under-predicts
the pool depths.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), layout='constrained', sharex=True)
for ax, smoothing in zip(axes, (0, 20, 60)):
    run = quick_run(mode='centerline',
                    centerline_file='../example_centerlines/jurua.txt',
                    width=160.0, depth=8.0, lag_strength=6.0,
                    smoothing=smoothing,
                    save_xyz=False, save_mesh=False, save_bankline=False,
                    save_figures=False, log_file='', verbose=False)
    s_hat = run.centerline.s / run.config.channel.width
    ax.axhline(0, color='0.85', linewidth=0.8)
    ax.plot(s_hat, run.curvature_original * run.config.channel.width, ':',
            color='darkorange', linewidth=0.9, label='original')
    ax.plot(s_hat, run.curvature_filtered * run.config.channel.width, '-',
            color='dodgerblue', linewidth=1.2, label='filtered')
    ax.set_ylabel('dimensionless\ncurvature')
    ax.set_title(f'smoothing level = {smoothing}', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(frameon=False, ncol=2)
axes[-1].set_xlabel('streamwise distance (channel widths)');

In [ ]:
jurua = quick_run(mode='centerline',
                  centerline_file='../example_centerlines/jurua.txt',
                  width=160.0, depth=8.0, lag_strength=6.0, smoothing=20,
                  save_xyz=False, save_mesh=False, save_bankline=False,
                  save_figures=False, log_file='', verbose=False)
print(jurua.summary())
show(jurua)

<a id="8"></a>
## 8. Meander migration and cutoffs

Switching migration on steps the planform forward with the linearised bend
theory of Ikeda, Parker & Sawai (1981). Bank retreat is proportional to the
near-bank excess velocity, which responds to both the local and the
phase-lagged curvature — and it is the lagged term that dominates, which is why
meanders both grow in amplitude and translate downstream.

`e0 * dt` is the displacement per step in channel widths, and it is the only
thing that sets the rate. Keep it well below 0.1 or the planform goes unstable.

Set `seed` to make a stochastic run reproducible.

In [ ]:
migrating = quick_run(
    n_bends=4, width=0.6, depth=0.15,
    migration=True, n_steps=600, dt=86400.0, e0=1e-7,
    neck_cutoff=True, seed=20210315,
    log_every=200, plot_every=200,
    output_dir='_nb_out', save_mesh=False, save_xyz=False,
    save_bankline=False, save_figures=False, save_gif=False, log_file='',
)
migrating.summary()

In [ ]:
show_planform(migrating)
show_diagnostics(migrating);

Sinuosity climbs while the bends grow, and drops abruptly whenever a cutoff
shortens the channel. That sawtooth is the signature of a meander belt in
dynamic equilibrium.

<a id="9"></a>
## 9. Chute cutoffs

Neck cutoffs are geometric and deterministic: they happen when a loop's two
limbs touch. **Chute cutoffs are different in kind** — the flow carves a new,
shorter channel across the floodplain while the limbs are still well separated,
and whether that happens depends on flood history, bank strength and bar
topography, none of which the model resolves.

pyRiverBed therefore treats them as *conditionally random*: the geometry decides
where a chute is **possible**, and a random draw decides whether one **happens**.

The geometric criteria a candidate must satisfy are:

| Parameter | Meaning |
|:---|:---|
| `chute_span` | how many entrance points the chute spans (2 = one meander loop) |
| `chute_min_length` | the bypassed reach must be this many channel widths long |
| `chute_min_sinuosity` | the bypassed reach must be this many times longer than the chute — the **slope advantage** |
| `chute_max_angle` | the chute chord must lie within this angle of the valley axis |
| `chute_start` | spin-up before any cutoff is allowed |
| `chute_frequency` | probability per time step, so calibrate as `dt / recurrence` |

In [ ]:
chutes = quick_run(
    n_bends=8, width=0.6, depth=0.15,
    migration=True, n_steps=400, dt=86400.0, e0=1e-7, seed=7,
    chute_cutoff=True, chute_frequency=0.15, chute_start=20,
    chute_span=2, chute_max_angle=45.0, chute_min_length=5.0,
    chute_min_sinuosity=1.2,
    log_every=200, plot_every=100,
    output_dir='_nb_out', save_mesh=False, save_xyz=False,
    save_bankline=False, save_figures=False, save_gif=False, log_file='',
)
print(chutes.summary())
for cut in chutes.cutoffs:
    print(f'  {cut.kind:6s} at step {cut.step:4d}: '
          f'nodes {cut.entrance}-{cut.exit}, '
          f'{cut.bypassed_nodes}-node oxbow lake')

In [ ]:
show_planform(chutes)

### The frequency sets the rate — until the bends run out

`chute_frequency` controls how *often* the model tries to carve a chute, which
is what makes it calibratable against an observed recurrence interval. But the
count is also **supply-limited**: every cutoff straightens the reach, and once
no bend still satisfies the geometric criteria, raising the frequency further
changes nothing.

Plotting the cumulative count against time shows both effects at once — a
steeper initial slope for a higher frequency, and every curve flattening as the
meander belt is consumed. In a long run the supply is replenished by continued
migration, so the two rates come back into balance; this short run deliberately
does not get that far.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), layout='constrained')
colours = {0.0: '0.6', 0.01: '#f39c12', 0.05: '#e74c3c', 0.2: '#8e44ad'}

for frequency, colour in colours.items():
    curves = []
    for seed in range(5):
        run = quick_run(
            n_bends=8, migration=True, n_steps=300, e0=1e-7, seed=seed,
            chute_cutoff=True, chute_frequency=frequency, chute_start=20,
            chute_max_angle=45.0, chute_min_length=5.0, chute_min_sinuosity=1.2,
            log_every=1000, plot_every=1000,
            output_dir='_nb_out', save_mesh=False, save_xyz=False,
            save_bankline=False, save_figures=False, save_gif=False,
            log_file='', verbose=False)
        steps = np.array([c.step for c in run.chute_cutoffs], dtype=int)
        curves.append(np.cumsum(np.bincount(steps, minlength=301))[:301])
    mean = np.mean(curves, axis=0)
    ax.plot(mean, color=colour, linewidth=2,
            label=f'frequency = {frequency}  (mean of 5 seeds)')
    ax.fill_between(np.arange(301), np.min(curves, axis=0),
                    np.max(curves, axis=0), color=colour, alpha=0.15)

ax.set_xlabel('time step')
ax.set_ylabel('cumulative chute cutoffs')
ax.set_title('Higher frequency cuts sooner, but the supply of bends is finite')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', color='0.92'); ax.set_axisbelow(True);

Two things to take from this:

* **`chute_frequency` is a rate, not a count.** Calibrate it as
  `dt / recurrence_interval` for the river you are modelling.
* **Results are one realisation of an ensemble.** The shaded bands above are
  the spread over five seeds at otherwise identical settings. Report ranges,
  not single runs.

<a id="10"></a>
## 10. Working with the raw arrays

Everything the model computed is on the result object, as plain NumPy arrays.
This is the point of the notebook frontend: you can take the synthetic bed and
do your own analysis with it.

In [ ]:
print('centerline :', type(result.centerline).__name__,
      f'({result.centerline.n_nodes} nodes)')
print('  .x .y .s .curvature .theta, and .length .sinuosity .valley_length')
print('bed.z      :', result.bed.z.shape, '(streamwise x transverse)')
print('cloud      :', result.cloud.shape, '(x, y, z point cloud)')
print()
print(f'reach length      {result.centerline.length:.3f} m')
print(f'sinuosity         {result.centerline.sinuosity:.4f}')
print(f'bar-pool relief   {result.bed.relief:.4f} m')
print(f'deepest pool      {result.bed.z.min():.4f} m')
print(f'highest bar       {result.bed.z.max():.4f} m')

In [ ]:
# Example: how does the transverse bed slope track the curvature?
fig, ax = plt.subplots(figsize=(11, 4), layout='constrained')
width = result.config.channel.width
ax.plot(result.centerline.s / width,
        result.curvature_lagged * width, color='orangered',
        label='lagged curvature (dimensionless)')
ax.plot(result.centerline.s / width, result.bed.transverse_slope,
        color='#2471a3', linestyle='--', label='transverse bed slope $S_T$')
ax.axhline(0, color='0.85', linewidth=0.8)
ax.set_xlabel('streamwise distance (channel widths)')
ax.set_title(r'$S_T = A H C \xi_{S_T}$ — the transverse slope is proportional'
             ' to the curvature')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False);

In [ ]:
# Example: interpolate the synthetic bed onto your own survey points.
from scipy.interpolate import griddata

points = result.cloud[:, :2]
values = result.cloud[:, 2]
survey_x = np.random.default_rng(0).uniform(points[:, 0].min(),
                                            points[:, 0].max(), 8)
survey_y = np.random.default_rng(1).uniform(points[:, 1].min(),
                                            points[:, 1].max(), 8)
elevations = griddata(points, values, (survey_x, survey_y), method='linear')
for sx, sy, sz in zip(survey_x, survey_y, elevations):
    inside = 'outside channel' if np.isnan(sz) else f'z = {sz:+.4f} m'
    print(f'({sx:7.3f}, {sy:7.3f})  {inside}')

<a id="11"></a>
## 11. Saving a configuration for the CLI or GUI

The three frontends read and write the same input file, so a configuration you
worked out in a notebook can be handed to a batch run or opened in the GUI.

In [ ]:
config = configure(n_bends=5, width=1.2, depth=0.25, migration=True,
                   n_steps=5000, chute_cutoff=True, chute_frequency=0.05)
prb.write_config(config, 'my_river.ini')
print(open('my_river.ini').read()[:700], '...')

Then, from a shell:

```bash
pyriverbed run my_river.ini          # batch
pyriverbed gui                       # open the GUI, then Open... my_river.ini
```

And back the other way:

In [ ]:
reloaded = prb.load_config('my_river.ini')
print('mode           ', reloaded.mode)
print('width          ', reloaded.channel.width)
print('chute frequency', reloaded.chute_cutoff.frequency)

<a id="12"></a>
## 12. Optional: an interactive form

If `ipywidgets` happens to be installed, `form()` gives you a small parameter
panel with a Run button. It is entirely optional — `ipywidgets` is **not** a
pyRiverBed dependency, and the cell below simply explains itself if it is
missing.

In [ ]:
from pyriverbed.notebook import form

panel = form()

## 13. Art prints

A migration run produces, almost incidentally, something you can hang on
a wall. `pyriverbed.art` draws the run as a **print** rather than a
figure: no axes, no legend, the channel as a filled ribbon of its own
width on a flat ground.

The `fisk` style is a homage to Harold Fisk's 1944 maps of the lower
Mississippi meander belt. Fisk drew every historical course of the river
in its own flat colour and let them overlap, which is exactly the data a
migration run leaves behind: `centerline_history` is a sequence of
courses, and every cutoff carries an oxbow lake stamped with the step it
was abandoned.

`fisk` and `strata` need a migration run so that there are historical
courses to layer. `bathymetry` and `contour` show the bed itself and zoom
in to do it, since at belt scale the channel is a hairline.

In [ ]:
from pyriverbed.notebook import art_styles, save_art, show_art

for name, description in art_styles().items():
    print(f'{name:11s} {description}')

# `chutes` is the migration run from section 9.
show_art(chutes, 'fisk')

In [ ]:
# Any style, any paper. `paper='fit'` (the default) shapes the canvas to
# the reach; a named size puts the print in a standard frame.
show_art(chutes, 'strata', title='My River', dpi=110)

# Write every style at print resolution. A .pdf or .svg name gives vector
# output, which is what you want for anything larger than a screen.
# save_art(chutes, 'prints', prefix='my_river', paper='a2', dpi=300)

---

## Where to go next

* **`THEORY_GUIDE.md`** — the physics: the Kinoshita curve, the Beck bed, the
  phase lag, the migration model, and both cutoff mechanisms.
* **`README.md`** — installation, the CLI, the GUI, and worked examples.
* **`pyriverbed.ini`** — every parameter with a one-line explanation.

### Citation

> Li, Z., & Garcia, M. H. (2021). pyRiverBed: A Python framework to generate
> synthetic riverbed topography for constant-width meandering rivers.
> *Computers & Geosciences*, 152, 104755.
> doi:[10.1016/j.cageo.2021.104755](https://www.doi.org/10.1016/j.cageo.2021.104755)

In [ ]:
# Tidy up the files this notebook wrote.
import shutil
from pathlib import Path

shutil.rmtree('_nb_out', ignore_errors=True)
Path('my_river.ini').unlink(missing_ok=True)
print('cleaned up')